# Week 7: Introductie GeoPandas in JupyterLite
In deze les leren we hoe we geografische data (kaarten, coördinaten en afstanden) kunnen verwerken met **GeoPandas** direct in de browser via **JupyterLite**.

## Deel 1: Packages installeren met `piplite`
In JupyterLite gebruiken we **`piplite`** om externe Python-packages direct vanuit Pyodide in te laden.

In [ ]:
import piplite

# GeoPandas, Matplotlib en Shapely installeren via piplite
await piplite.install(['geopandas', 'matplotlib', 'shapely'])

## Deel 2: Handmatig een GeoDataFrame maken
Een **GeoDataFrame** is een Pandas DataFrame met een speciale `geometry` kolom die `Point`, `LineString` of `Polygon` objecten bevat.

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Data met steden en hun coördinaten (Lengtegraad / Breedtegraad)
data = {
    "Stad": ["Amsterdam", "Utrecht", "Rotterdam", "Groningen"],
    "Inwoners": [872000, 360000, 650000, 233000],
    "Lat": [52.3676, 52.0907, 51.9244, 53.2194],
    "Lon": [4.9041, 5.1214, 4.4777, 6.5665]
}

df = pd.DataFrame(data)

# Shapely Points maken (let op de volgorde: Point(longitude, latitude))
geometry = [Point(lon, lat) for lon, lat in zip(df["Lon"], df["Lat"])]

# GeoDataFrame maken met WGS84 coördinatenstelsel (EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf

## Deel 3: Kaart Visualiseren
Met de `.plot()` methode kun je direct ruimtelijke punten op een kaart laten zien.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="red", markersize=gdf["Inwoners"]/10000, alpha=0.7)

# Labels toevoegen per stad
for x, y, label in zip(gdf.geometry.x, gdf.geometry.y, gdf["Stad"]):
    ax.text(x + 0.05, y, label, fontsize=10)

plt.title("Nederlandse Steden (Grootte = Inwoners)")
plt.xlabel("Lengtegraad (Longitude)")
plt.ylabel("Breedtegraad (Latitude)")
plt.grid(True)
plt.show()

## Deel 4: Afstanden berekenen in kilometers
Om afstanden in kilometers te berekenen, transformeren we het coördinatensysteem naar het Nederlandse Rijksdriehoeksstelsel (`EPSG:28992`).

In [ ]:
# Omzetten naar het Nederlandse RD New stelsel (meters)
gdf_rd = gdf.to_crs(epsg=28992)

# Afstand berekenen vanaf Amsterdam (index 0) in km
amsterdam_geom = gdf_rd.geometry.iloc[0]
gdf["Afstand_tot_Amsterdam_km"] = round(gdf_rd.geometry.distance(amsterdam_geom) / 1000, 1)

gdf[["Stad", "Afstand_tot_Amsterdam_km"]]

## Deel 5: Geodata Exporteren & Inlezen (GeoJSON)
Sla ruimtelijke data op als **GeoJSON**.

In [ ]:
# Exporteren naar GeoJSON
gdf.to_file("steden.geojson", driver="GeoJSON")
print("Bestand 'steden.geojson' is succesvol opgeslagen!")

# Inlezen vanuit GeoJSON
ingelezen_gdf = gpd.read_file("steden.geojson")
ingelezen_gdf